<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 El Traductor de la Plaza: Convertir Palabras en Números 🔤
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 06
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Estadistica%20analisis%20y%20representacion%20de%20datos/06%20-%20Modelamiento%20Estadistico/Para%20Dummies/00_Codificacion_Variables_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 00 del Módulo 06 (Representación de datos)](../00_Representacion_de_Datos_OneHot_y_Label_Encoding.ipynb). Una calculadora solo entiende **números**, pero tus datos dicen cosas como «Pastusa», «Criolla» o «vereda La Esperanza». Hay que **traducirlas**, y la traducción importa más de lo que parece: una mala traducción hace que el modelo **invente cosas**.

| En la plaza de mercado | En estadística |
|---|---|
| Numerar las variedades de papa 1, 2, 3, 4 como si fueran una fila | **Codificación por etiquetas** (*label encoding*) |
| Una casilla de «sí / no» por cada variedad | **Una casilla por categoría** (*one-hot*) |
| «Si no es de A, B ni C, entonces es de D»: la última casilla sobra | La **categoría de referencia** |
| Copiarse del cuaderno donde ya está la respuesta | **Fuga de información** (mirar la respuesta al traducir) |

Al terminar podrás explicar:
1. Por qué **numerar** variedades (1, 2, 3, 4) **engaña** a un modelo.
2. Cómo funcionan las **casillas de sí/no** y por qué **una sobra**.
3. Por qué **no hay que mirar la respuesta** cuando se traduce una categoría con muchos valores.

> 🔗 Si luego quieres ver las versiones **con fórmulas, pruebas y simulaciones**: [cuaderno estándar](../00_Representacion_de_Datos_OneHot_y_Label_Encoding.ipynb). Taller del módulo: [`../../homeworks/Para Dummies/06_Modelamiento_Estadistico_Hands_On_Dummies.ipynb`](../../homeworks/Para%20Dummies/06_Modelamiento_Estadistico_Hands_On_Dummies.ipynb).

---
## 1. Numerar las Variedades Engaña 🔢

Tenemos 200 fincas **inventadas** con cuatro variedades de papa. El rendimiento promedio de cada una es muy distinto, **pero no sigue el orden alfabético**. Si numeras las variedades por orden alfabético (Capiro = 0, Criolla = 1, Pastusa = 2, Sabanera = 3) y le pides al modelo una **línea recta**, el modelo cree que Criolla está «entre» Capiro y Pastusa y que de una a otra hay siempre el mismo salto. Veamos qué tan mal predice cada variedad.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# 200 fincas INVENTADAS con 4 variedades; el rendimiento promedio verdadero de cada una (t/ha) NO sigue el orden alfabetico
verdad = {"Capiro": 21, "Criolla": 18, "Pastusa": 25, "Sabanera": 20}
fincas = pd.DataFrame({"variedad": rng.choice(list(verdad), 200)})
fincas["rend"] = fincas["variedad"].map(verdad) + rng.normal(0, 2.5, 200)

# Traduccion 1: numerar por orden alfabetico y ajustar UNA recta
codigo = {"Capiro": 0, "Criolla": 1, "Pastusa": 2, "Sabanera": 3}
fincas["num"] = fincas["variedad"].map(codigo)
pend, orig = np.polyfit(fincas["num"], fincas["rend"], 1)
fincas["pred_recta"] = orig + pend * fincas["num"]

# Traduccion 2: usar el promedio real de cada variedad (lo que logra una casilla por variedad)
fincas["pred_casillas"] = fincas.groupby("variedad")["rend"].transform("mean")

resumen = fincas.groupby("variedad").agg(real=("rend", "mean"), recta_con_numeros=("pred_recta", "mean"), con_casillas=("pred_casillas", "mean")).round(1)
print(resumen.to_string())
err_recta = np.sqrt(np.mean((fincas["rend"] - fincas["pred_recta"]) ** 2))
err_casillas = np.sqrt(np.mean((fincas["rend"] - fincas["pred_casillas"]) ** 2))
print(f"\nError tipico (t/ha): con numeros = {err_recta:.2f} | con casillas = {err_casillas:.2f}")
assert err_recta > 1.3 * err_casillas

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(codigo.values(), resumen["real"], s=120, color="#f59e0b", zorder=3, label="promedio real")
ax.plot(range(4), orig + pend * np.arange(4), color="#dc2626", lw=2, label="lo que cree la recta")
ax.set_xticks(range(4)); ax.set_xticklabels(codigo.keys()); ax.set_ylabel("rendimiento (t/ha)"); ax.legend()
ax.set_title("La recta supone un orden que no existe"); plt.show()

---
**Lección:** con los números 0, 1, 2, 3, la recta **no puede** darle a Criolla un valor bajo y a Pastusa uno alto al mismo tiempo: queda cerca de ninguno (error típico ≈ 3.6 t/ha). Con **una casilla por variedad**, cada variedad recibe **su propio promedio** y el error baja a ≈ 2.5 t/ha, que es el ruido natural de las fincas. Moraleja: **numerar categorías sin orden real le inventa un orden al modelo**.

---
## 2. Una Casilla de Sí/No por Variedad (y Una Sobra) ☑️

La traducción correcta crea **una casilla por variedad**: «¿es Pastusa? sí/no», «¿es Criolla? sí/no»... Pero fíjate: si ya sabes que **no** es Criolla, **ni** Pastusa, **ni** Sabanera, entonces **tiene que ser** Capiro. **La última casilla sobra.** La que dejamos fuera se llama la **referencia**, y las demás casillas dicen **cuánto más o menos** rinde esa variedad que la de referencia.

In [ ]:
casillas = pd.get_dummies(fincas["variedad"], dtype=int)
print(casillas.head(4).to_string(), "\n")
assert (casillas.sum(axis=1) == 1).all()                  # cada finca tiene exactamente un "1": la ultima casilla se puede deducir de las otras

# Dejamos Capiro como referencia: nos quedan 3 casillas. El "modelo" es: promedio de Capiro + salto de cada variedad
ref = "Capiro"
medias = fincas.groupby("variedad")["rend"].mean()
saltos = (medias - medias[ref]).drop(ref).round(2)
print(f"Referencia = {ref}: rinde en promedio {medias[ref]:.1f} t/ha")
for v, s in saltos.items():
    print(f"   {v:9s} rinde {s:+.2f} t/ha respecto a {ref}")

# Cambiar la referencia cambia los saltos, pero NO las predicciones
ref2 = "Pastusa"
saltos2 = (medias - medias[ref2]).drop(ref2).round(2)
pred1 = np.array([medias[ref] + (saltos[v] if v != ref else 0) for v in fincas["variedad"]])
pred2 = np.array([medias[ref2] + (saltos2[v] if v != ref2 else 0) for v in fincas["variedad"]])
print(f"\nCon referencia {ref2}: {saltos2.to_dict()}")
assert np.allclose(pred1, pred2, atol=0.02)               # mismas predicciones (hasta el redondeo)
print("Las predicciones son las mismas con cualquier referencia: solo cambia la forma de CONTAR la historia.")

---
**Lección:** el modelo con casillas es solo «el promedio de la variedad de referencia + el salto de cada una»; con otra referencia los saltos cambian de signo y tamaño, pero las predicciones son las mismas.

---
## 3. No Copies de la Hoja de Respuestas 📝

A veces la categoría tiene **muchísimos valores** (60 veredas con 5 fincas cada una). Una idea tentadora: reemplazar cada vereda por **el promedio de rendimiento de sus fincas**. Pero si ese promedio se calcula **usando la respuesta de la misma finca**, estás **copiando de la hoja de respuestas**: el modelo luce brillante con las fincas que ya vio y falla con fincas nuevas. Simulamos un caso donde **la vereda no influye en nada** (todo es azar) para ver la trampa.

In [ ]:
rng = np.random.default_rng(3)
def generar(n=300):
    return pd.DataFrame({"vereda": rng.integers(0, 60, n), "rend": rng.normal(20, 3, n)})     # la vereda NO influye: es puro azar

entrenamiento, prueba = generar(), generar()
promedio_vereda = entrenamiento.groupby("vereda")["rend"].mean()                       # promedio calculado CON la respuesta de cada finca

copiado_entr = entrenamiento["vereda"].map(promedio_vereda)
copiado_prueba = prueba["vereda"].map(promedio_vereda).fillna(entrenamiento["rend"].mean())
r_entr = np.corrcoef(copiado_entr, entrenamiento["rend"])[0, 1]
r_prueba = np.corrcoef(copiado_prueba, prueba["rend"])[0, 1]
print(f"Parecido (correlacion) entre la 'pista' y el rendimiento real: fincas que ya vio = {r_entr:.2f} | fincas nuevas = {r_prueba:.2f}")
assert r_entr > 0.3 and abs(r_prueba) < 0.2

---
##### 🛠️ Práctica 1: Tu turno: la referencia que tú eliges

Con `fincas` (200 fincas) y los promedios `medias` de la sección 2:

1. Elige `"Sabanera"` como referencia y calcula `saltos_sab`: el promedio de cada variedad **menos** el promedio de Sabanera (sin la propia Sabanera).
2. Verifica con `assert` que `saltos_sab["Pastusa"]` es **positivo** (Pastusa rinde más que Sabanera) y que `saltos_sab["Criolla"]` es **negativo**.
3. Comenta con tus palabras: ¿qué te dice el número de `Pastusa`?

In [ ]:
# Escribe tu código aquí

# saltos_sab = ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
saltos_sab = (medias - medias["Sabanera"]).drop("Sabanera")
print(saltos_sab.round(2).to_dict())

assert saltos_sab["Pastusa"] > 0 and saltos_sab["Criolla"] < 0
print("Pastusa rinde unas 5 t/ha mas que Sabanera, en promedio.")
```
</details>

---


---
**Lección:** con las fincas que el modelo **ya vio**, la «pista» parece muy buena (correlación ≈ 0.4 **inventada** por haber copiado la respuesta); con fincas **nuevas**, no sirve para nada (≈ 0). Regla: **cuando traduzcas usando la respuesta, hazlo sin mirar la finca que estás traduciendo** (o con otro grupo de fincas). Los modelos serios lo hacen automáticamente en varias rondas.

---
## Resumen en una frase 🎯

> **Para que un modelo entienda palabras hay que traducirlas a números sin inventar un orden que no existe (una casilla de sí/no por categoría, dejando una de referencia) y sin copiar de la hoja de respuestas.**

### Lo que aprendiste hoy:
- ✅ **Numerar** variedades (0, 1, 2, 3) le **inventa un orden** al modelo.
- ✅ Una **casilla por categoría** deja que cada una tenga su propio promedio.
- ✅ **Una casilla sobra**: la que dejas fuera es la **referencia**, y las demás cuentan «cuánto más o menos» que ella.
- ✅ Cambiar la referencia **no cambia las predicciones**, solo la forma de contarlas.
- ✅ Traducir con la **respuesta de la misma finca** es **copiar**: luce bien con lo conocido y falla con lo nuevo.

---
### 🧠 ¿Ya lo tienes claro?

1. Un amigo numera los municipios de Boyacá del 1 al 123 por orden alfabético y los mete como una sola columna. ¿Qué «orden» le estás inventando al modelo? ¿Cómo lo arreglarías?
2. Si tienes 5 variedades de papa, ¿cuántas casillas de sí/no necesitas realmente? ¿Por qué?
3. Explica con una analogía de colegio por qué traducir una categoría con la respuesta de la misma finca es hacer trampa.

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos (Edición Didáctica Para Dummies)</i>
  </p>
</div>